FILE 1: `policy.py` -- THE CONTRACT

LINES 90-134 -- `set_pose_target()` -- THE CONVENIENCE HELPER

```Python
    def set_pose_target(
        self, move_robot: MoveRobotCallback, pose: Pose, frame_id: str = "base_link"
    ) -> None: 
```
   This wraps up a `MotionUpdate` message with sensible defaults. The key 
   parameters it sets:

```Python
    motion_update = MotionUpdate(
        target_stiffness=...,
        target_damping=...,
    )
```
   * STIFFNESS ...: How strongly the arm tracks the target pose. First 3 = 
     translational (N/m), last 3 - rotational (Nm/rad). Higher = stiffer 
     tracking, more jerk.
   * DAMPING ...: Velocity-proportional resistance. Higher = smoother but slower

```Python
    wrench_feedback_gains_at_tip = ... 
```
    Force feedback gains. [0.5, 0.5, 0.5] means 50% force feedback on 
    translation (the arm yields to external forces). [0, 0, 0] on rotational
    means no rotationalc ompliance. Range is [0, 0.95].

```Python
    trajectory_generation_mode=TrajectoryGenerationMode(
        mode=TrajectoryGenerationMode.MODE_POSITION,
    ),
```
   `MODE_POSITION` means the pose field is used. `MODE_VELOCITY` means the
   velocity field is used instead.


LINES 136-145 -- THE ABSTRACT METHOD YOU IMPLEMENT
```Python
    @abstractmethod
    def insert_cable(
        self,
        task: Task,                                 # What to insert where
        get_observation: GetObservationCallback,    # See the world
        move_robot: MoveRobotCallback,              # Command the arm
        send_feedback: SendFeedbackCallback,        # Report progress
    ) -> bool:                                      # True = success
```



---
FILE2: `aic_model.py` -- THE LIFECYCLE NODE (Your Runtime)
   PATH: `aic_model/aic_model/aic_model.py` (335 lines)
   IMPORTANCE: Understand this to know waht happens around your code.

KEY LINES EXPLAINED

Lines 53-80 -- Dynamic Policy Loading

```Python
class AicModel(LifecycleNode):
    def __init__(self):
        super().__init__("aic_model")
        self.declare_parameter("policy", "WaveArm")
        policy_module_name = (
            self.get_parameter("policy").get_parameter_value().string_value
        )
        policy_module = importLib.import_module(policy_module_name)
```
   Your policy is loaded dynamically via string name. When you run 
   `ros2 run aic_model aic_model --ros-args -p policy:=my_package.MyPolicy`,
   it imports `my_package.MyPolicy` and looks for a class named `MyPolicy` 
   inside.

   
LINES 81-84 -- TF BUFFER SETUP

```Python
    self._tf_buffer = Buffer()
    self._tf_listener = TransformListener(
        buffer=self._tf_buffer, node=self, spin_thread=True
    )
```
   The TF buffer accumulates transform data from `/tf` and `/tf_static`. Your
   policy accesses this via `self._parent_node._tf_buffer`. When 
   `ground_truth:=true`, cable and port TF frames are published here.


LINES 91-93 -- OBSERVATION SUBSCRIPTION
```Python
    self.observation_sub = self.create_subscription(
        Observation, "observations", self.observation_callback, 10
    )
```
   Subscribes to the `/observations` topic (published by `aic_adapter` at 20Hz).
   When your policy calls `get_observation()`, it gets whatever was last stored.


LINES 190-229 -- AUTOMATIC MODE SWITCHING
```Python
    def handle_motion_update(self, motion_update: MotionUpdate):
        if self._target_mode != TargetMode.MODE_CARTESIAN:      # Line 191
            self.set_target_mode(TargetMode.MODE_CARTESIAN)     # Line 193
        self.motion_update_pub.publish(motion_update)    
```
   If you send a `MotionUpdate` but your controller is in joint mode, it 
   automatically calls the `ChangeTargetMode` service to switch.


LINES 236-244 -- HOW YOUR POLICY GETS CALLED
   Your `insert_cable()` runs in a separate thread (`action_thread_func`). The
   main thread continues spinning ROS callbacks. This is why `get_observation()`
   always returns the latest data even while your policy is in a loop.


---
FILE 3: `Observation.msg` -- EVERYTHING YOU CAN SEE
   PATH: `aic_interfaces/aic_model_interfaces/msg/Observation.msg`
   IMPORTANCE: This is your sensor input -- every perception decision depends on
   this.








---

    * THE SUPREME ENTRY POINT: ./aic/aic_model/aic_model/aic_model.py
            - This contains the main function, and it has a series of scalpers
              from lines 62-122 which extracts the various input parameters
              given from the entry command.
`ros2 run aic_model aic_model --ros-args -p policy:=aic_example_policies.ros.CheatCode`
            - aic_model.py's AicModel(LifecycleNode) class then scalps the class
              definition of `aic_example_policies.ros.CheatCode` in line 67
              `inspect.getmembers(policy_module, inspect.isclass)`.
            - The `CheatCode.py`'s ckass is defined as CheatCode(policy) --
              which implements from the interface in 
              `./aic/aic_model/aic_model/policy.py`... and hence the full circle
              is reached and everything ties together nicely.
          
            - Which all the client-side (our side)'s configs are passed in from
              `class CheatCode(policy)` ... which this is where we implement the
              list of 10 policy models, and other fun stuff too...
              

the main entry point seems to be `aic_model.py` the policy interface file is#
`policy.py`

our custom policy can be implemented anywhere in the directory (though our 
custom policy class itself implements from the interface class `Policy` from
`Policy.py`)


---

    But in the ROS2 command that we shoved into terminal: 
`ros2 run aic_model aic_model --ros-args -p policy:=aic_example_policies.ros.CheatCode`

    This tells `aic_model.py` to treat `aic_examnple_policies/ros/CheatCode.py`
    as the policy .... 
    
    ...


---

    so yeah, the rest are just `.msg` files which are intermediary kinda 
    serialised objects pass between functions or stored during operation

    or template examples files for ACT (action chunking with transformers <-- I
    wonder can we change to VLA from here or sth... but likely not useful 
    because we have a dead fixed task here...), or the cheat code file... which
    from what im seeing basically are just: `RunACT.py` 









---

PART 1: WHAT EXACTLY IS AN "ACT"?
    ACT stands for ACTION CHUNKING WITH TRANSFORMERS. It is currently one of the
    most popular and powerful Imitation Learning (IL) architectures for 
    real-world robotic manipulation, originally developed...

    To understand why it's a breakthrough, you have to understand the problem
    it solves:
    
   1. THE COMPOUNDING ERROR PROBLEM: Older AI models predicted robot movement
      one single step at a time (e.g., "move 1mm left"). If the robot made a 
      tiny mistake, the next camera frame would look slightly weird, causing it
      to make a bigger mistake on the next step. The robot would quickly jitter
      and fail.
   2. THE "CHUNKING" SOLUTION: Instead of predicting 1 step, ACT predicts a 
      "CHUNK" of future actions--usually the next 50 to 100 steps all at once.
   3. TEMPOLAR ENSEMBLING: At step 1, it predicts steps 1-100. At step 2, it
      predicts steps 2-101. It overlaps these predictions and averages them out.
      This makes the robot's movements incredibly smooth and dramatically 
      reduces jitter and errors.
   4. THE TRANSFORMER: It uses a Transformer (similar architecture to LLMs) to
      process multiple camera viewpoints (left, center, right) and the robot's
      join states simultaneously to understand the 3D spatial relationship of
      the scene.

   IN SHORT: You teleoperate the robot to insert the cable 50 times. You feed 
   that data into ACT. ACT learns to mimic your exact fluid, multi-step motions
   using camera feeds.


---
PART 2: DECONSTRUCTING `RunACT.py`
    This file is a bridge. It takes the AI model you trained using Hugging 
    Face's `lerobot` library and plugs it into the Intrinsic AI Challenge
    simulator.

    Here is what the code is doing, step-by-step:

   1. THE SETUP && LOADING (Lines 55-90)
       * WHAT IT DOES: It connects to the HF internet hub, downloads a specific
         pre-trained ACT model (`grkw/aic_act_policy`), and loads the neural
         network weights into your GPU (`self.device = torch.device("cuda")`).
       * WHY IT MATTERS: This means you don't have to package gigabytes of model
         weights into your Docker container. It pulls your brain from the cloud
         at runtime. 
   2. THE GREAT NORMALISATION (Lines 91-168)
       * WHAT IT DOES: AI models don't understand raw pixels (0-255) or raw 
         robot coordinates. They need numbers squashed between roughly -1 and 1.
         This section loads the MEAN and STANDARD DEVIATION (std) of the data
         you used during training, and applies the formula `(x - mean) / std` to
         the live camera and robot data. 
       * WHY IT MATTERS: If you skip this, or use the wrong stats, your robot 
         will flail wildly becuse the input numbers look completely alien to the
         neural network.
   3. DATA FORMATTING (Lines 169-236)
      * WHAT IT DOES: The `prepare_observations` function takes the nice ROS
        `Observation.msg` and shreds it into raw NumPy/PyTorch arrays. It stacks
        the TCP position, orientation, velocities, and joint states into one
        massive 26-dimensional flat array.             
   4. THE INFERENCE LOOP (Lines 237-296)
      * WHAT IT DOES: This is the `insert_cable` function that runs the trail.
         * It grabs the cameras/state (`get_observation()`).
         * Prepares the data.
         * LINE 266 IS THE MAGIC: 
            `normalized_action = self.policy.select_action(obs_tensor)`
           The AI looks at the cameras and spits out what to do next.
         * It un-normalises the output back into real-world speeds.
         * It extracts a 6D velocity vector (Linear X/Y/Z + Angular X/Y/Z)
         * It loops at 4Hz (`time.sleep(0.25)`).
   5. THE COMMAND EXECUTION (Lines 297-321)
      * WHAT IT DOES: It packages that 6D velocity into a `MotionUpdate` message
      * CRUCIAL DETAIL: Notice on L317 it sets 
        `mode=TrajectoryGenerationMode.MODE_VELOCITY`. Unlike the CheatCode
        policy which commands positions, ACT commands the robot's speed and
        direction, riding it like a joystick.


---
PART 3: WHEN, HOW, AND WHY TO USE THIS FILE

WHEN WILL YOU USE IT?
   You will use this file IF YOU DECIDE TO USE IMITATION LEARNING for the 
   competition of your hackathon.

   If your plan is: "I'm going to set up my Xbox controller or VR headset,
   teleoperate the simulated robot to plug in the SFP cable 100 times, and have
   an AI copy me" -- this is exactly the file you will use to run that AI.

HOW WILL YOU MODIFY IT?

   1. LINE 62 (`repo_id = "grkw/aic_act_policy"`): You will change this to point
      to your HF account (e.g., ...) ...

   2. LINE 198-228 (`state_np = np.array(...)`): If you decide you only want
      your AI to look at TCP positions and ignore joint states to make training
      faster, you will change this array to match whatever you trained on.

   3. LOGIC ROUTING: As we discussed earlier, you might add an `if` statement in
      `insert_cable` to load a different `repo_id` depending on if 
      `task.plug_type` is SFP or SC.

   WHY US IT OVER OTHER METHOD?      
      * PROS OF ACT: It is phenomenally good at contact-rich tasks (like 
        sldiding a plug into a tight port). It handles the "wiggle" natually if 
        your human demonstrations include wiggling. It doesn't require compleax 
        math or coding the exact physics of the port.
      * CONS OF ACT: You have to collect the data manually. Gathering 100
        successful human demonstrations of cable insertion in simulation can
        take hours of tedious, repetitive work.

   WHY NOT USE IT?
      If you decide to solve this using RL (where the robot learns by trail and
      error over millions of simulation steps) or CLASSICAL CONTROL (using 
      OpenCV to find the port and writing math to calculate the perfect 
      trajectory, like `CheatCode.py` does), you will ignore `RunACT.py` 
      entirely.




                    


